In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
tracking_df = spark.read.format("delta").table("dev.tracking_sl.ordertracking")

In [0]:
dimdate_df = spark.read.format("delta").table("dev.tracking_gd.dimdate")

In [0]:
tracking_df = tracking_df.select(col('shipmentno').alias('shipmentno'),col('orderstatus').alias('status'),col('trackingtime'))

In [0]:
tdetail_df = tracking_df.join(dimdate_df, to_date(tracking_df.trackingtime) == dimdate_df['date'], "inner")

In [0]:
from pyspark.sql.functions import max
tdetail_df = tdetail_df.groupby('shipmentno','status').agg(max('datekey').alias('datekey'))



In [0]:
tdetail_df = tdetail_df.select('shipmentno','status','datekey')\
    .withColumn('ingestion_time', current_timestamp())

In [0]:
tdetail_df.createOrReplaceTempView("temp_tracking")

In [0]:
%sql
merge into dev.tracking_gd.dimtracking tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_tracking 
) src 
on tgt.shipmentno = src.shipmentno
and tgt.status = src.status
when not matched then
  insert (shipmentno, datekey, status, ingestion_time)
  Values(src.shipmentno,src.datekey,src.status,src.ingestion_time)

In [0]:
%sql
select * From dev.tracking_gd.dimtracking limit 1000